# 在 Colab GPU 上继续 CQI × 压缩实验

本笔记本在 Colab 运行时上克隆仓库、安装环境、做 GPU 自检，然后运行 `cqi_compression/run_cqi_bler.py` 并汇总。实验设计、进度和注意事项见 [README.md](README.md) 与 [experiments.md](experiments.md)。

**准备**
1. 运行时类型选 GPU（L4 / A100 / T4 均可）。
2. 仓库目前是公开的，第 3 步直接克隆，不需要令牌。若改回私有：把第 1 步的 `PRIVATE_REPO` 设为 True，并准备一个只读的 fine-grained token（Repository access 选中本仓库，Contents: Read-only），放进 Colab Secrets 的 `GITHUB_TOKEN`，读不到时会提示输入。

**在 VS Code 中使用**：用 Colab 扩展把内核连接到 Colab 运行时后，从上到下依次运行。笔记本文件留在本地，单元格输出（汇总表、图）随笔记本保存在本地；运行时里的文件在断开后会丢失，所以结果写到 Google Drive（`drive.mount` 不可用时见第 2 步的说明）。

以下单元格未在 Colab 上实际运行过，第一次使用时请留意输出。

In [ ]:
# 1. 参数
REPO = "Agata872/Sionna-based-Simulation"
BRANCH = "main"
PRIVATE_REPO = False   # 仓库已公开；改回私有时设为 True，第 3 步会读取只读令牌
WORKDIR = "/content/Sionna-based-Simulation"
DRIVE_RESULTS = "/content/drive/MyDrive/sionna_results/cqi_compression_20261029"   # 结果写到 Google Drive
LOCAL_RESULTS = "/content/results/cqi_compression_20261029"                       # Drive 不可用时的退路
DEVICE = "cuda:0"     # GPU 自检不通过时改为 "cpu"
RT_LLVM = False       # 射线追踪报 OptiX 错误时改为 True（射线追踪改用 CPU，链路仍在 GPU）

In [ ]:
# 2. 挂载 Google Drive，结果写到 Drive；挂载失败时写到运行时本地磁盘（断开后丢失，需要及时下载）
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS = DRIVE_RESULTS
except Exception as e:
    print(f"Drive 挂载失败（{type(e).__name__}: {e}），结果写到 {LOCAL_RESULTS}")
    print("可在浏览器中打开同一运行时挂载 Drive，或运行结束后把该目录打包下载")
    RESULTS = LOCAL_RESULTS
os.makedirs(RESULTS, exist_ok=True)
print("结果目录：", RESULTS)

In [ ]:
# 3. 克隆或更新仓库。公开仓库直接克隆；私有仓库的令牌只用于本次命令，不写入 .git/config，输出和报错中都不出现令牌
import subprocess, getpass
token = None
if PRIVATE_REPO:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass.getpass("GitHub token（只读）：")
    token = token.strip() or None                      # 复制时可能带上空格或换行
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

def git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    out = r.stdout + r.stderr
    if token:
        out = out.replace(token, "***")
    print(out)
    if r.returncode:
        hint = ""
        if "403" in out:
            hint = ("\n403：令牌有效但没有读取本仓库的权限。到 GitHub 的 Fine-grained tokens 页面编辑该令牌："
                    "Repository access 选中本仓库，Repository permissions 中 Contents 设为 Read-only。")
        elif "401" in out or "Authentication failed" in out:
            hint = "\n401：令牌无效（复制不完整、已过期或已作废），请重新生成。"
        raise RuntimeError(f"git {args[0]} 失败（退出码 {r.returncode}）{hint}") from None

if not os.path.isdir(WORKDIR):
    git("clone", "--depth", "1", "--branch", BRANCH, url, WORKDIR)
    git("remote", "set-url", "origin", f"https://github.com/{REPO}.git", cwd=WORKDIR)
else:
    git("pull", url, BRANCH, cwd=WORKDIR)
os.chdir(WORKDIR)
git("log", "--oneline", "-3", cwd=WORKDIR)


In [ ]:
# 4. 安装依赖：保留 Colab 自带的 torch（Sionna 2.2.0 要求 torch >= 2.9.1），只装 Sionna 与中文字体
!pip install -q sionna==2.2.0 sionna-rt==2.2.0
!apt-get -qq install -y fonts-noto-cjk > /dev/null && rm -rf ~/.cache/matplotlib

In [ ]:
# 5. 环境检查：GPU、torch、Mitsuba 变体（cuda_ad_* 表示射线追踪在 GPU 上）
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!python -c "import torch, sionna.phy, mitsuba as mi, sionna.rt; print('torch', torch.__version__, '| CUDA', torch.version.cuda, '| GPU 可用', torch.cuda.is_available(), '| Mitsuba', mi.variant())"

In [ ]:
# 6. GPU 自检（约 2 分钟）：两项都 PASS 才用 GPU；FAIL 时把第 1 步的 DEVICE 改为 "cpu"
!python cqi_compression/gpu_selfcheck.py --device {DEVICE}

In [ ]:
# 7. 冒烟测试（几十秒）：检查流程能跑通、复核一致
RT = "--rt-llvm" if RT_LLVM else ""
!python cqi_compression/run_cqi_bler.py --device {DEVICE} {RT} --drops 2 --periods 1 --widths ideal 5 \
    --out {RESULTS}/smoke_colab 2>&1 | grep -v "Warning\|LDPC5GEncoder"

In [ ]:
# 8. 正式运行：50 个撒点 × Cat A/B × 8 个位宽 × 4 个周期。
#    GPU 预计约 1 小时（本地 RTX 4060 冒烟测试的速度外推），CPU 约 3.5 小时。
#    运行目录已存在 rows.jsonl 时脚本会拒绝覆盖：中断后重跑请换一个 --out 名字（如 main2）。
OUT = f"{RESULTS}/main"
!python cqi_compression/run_cqi_bler.py --device {DEVICE} {RT} --drops 50 --out {OUT} 2>&1 \
    | grep -v "Warning\|LDPC5GEncoder" | tee {RESULTS}/main.log

In [ ]:
# 9. 汇总：表格与图（输出随笔记本保存在本地）。报告开头若有"复核不一致"警告，本次结果作废
!python cqi_compression/analyse_cqi_bler.py {OUT}
from IPython.display import Image, display
display(Image(f"{OUT}/fig_bler_tput_vs_width.png"))

In [ ]:
# 10. 运行时本地保存时，打包结果以便下载（Drive 上的结果无需此步）
if RESULTS == LOCAL_RESULTS:
    !cd /content/results && zip -qr /content/cqi_compression_20261029.zip cqi_compression_20261029
    print("已打包：/content/cqi_compression_20261029.zip")